In [ ]:
import os
import sys
import shutil
import zipfile
from pathlib import Path
from tqdm import tqdm

In [ ]:
# ============================================================================
# 1. TẢI CODE TỪ GITHUB
# ============================================================================
REPO_URL = "https://github.com/hotien2107/mossformer-mamba.git"
REPO_DIR = "/content/mossformer-mamba"

if os.path.exists(REPO_DIR):
    print(f"Đang xóa thư mục cũ...")
    shutil.rmtree(REPO_DIR)

print(f"Đang tải code từ {REPO_URL}...")
!git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

Đang tải code từ https://github.com/hotien2107/mossformer-mamba.git...
Cloning into '/content/mossformer-mamba'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 137 (delta 56), reused 108 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 783.87 KiB | 13.06 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/mossformer-mamba


In [ ]:
# ============================================================================
# 2. CẤU HÌNH ĐƯỜNG DẪN
# ============================================================================
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = Path("/content/mossformer-mamba/mossformer2")
DATA_ROOT = Path("/content/datasets/VN_SpeechMix_8s_8khz")
ZIP_PATH = Path("/content/drive/MyDrive/VN-SpeechMix_Datasets/dataset_8s_8khz.zip")
CKPT_DIR = Path("/content/drive/MyDrive/checkpoint-mossformer-lite-8s-8khz")
DATA_DIR = WORK_DIR / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
# ============================================================================
# 3. GIẢI NÉN DATASET
# ============================================================================
if not DATA_ROOT.exists() or not list(DATA_ROOT.glob("train/mix/*.wav")):
    print(f"\n📦 Đang giải nén dataset...")
    os.makedirs(DATA_ROOT, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, 'r') as zipf:
        files = zipf.namelist()
        for f in tqdm(files, desc="Giải nén", unit="files"):
            zipf.extract(f, DATA_ROOT)
    print("✅ Giải nén hoàn tất!")
else:
    print(f"✅ Dataset đã có sẵn")


📦 Đang giải nén dataset...


Giải nén: 100%|██████████| 60754/60754 [02:29<00:00, 405.97files/s]

✅ Giải nén hoàn tất!


In [ ]:
# ============================================================================
# 4. TẠO FILE SCP
# ============================================================================
print(f"\n📝 Tạo file SCP...")

split_map = {
    "train": ("train", "train.scp"),
    "valid": ("valid", "dev.scp"),
    "test":  ("test", "test.scp"),
}

for orig_split, (data_split, scp_name) in split_map.items():
    mix_dir = DATA_ROOT / data_split / "mix"
    s1_dir = DATA_ROOT / data_split / "s1"
    s2_dir = DATA_ROOT / data_split / "s2"

    if not mix_dir.exists():
        continue

    files = sorted(mix_dir.glob("*.wav"))
    lines = []

    for f in files:
        s1 = s1_dir / f.name
        s2 = s2_dir / f.name
        if s1.exists() and s2.exists():
            lines.append(f"{f} {s1} {s2}\n")

    scp_path = DATA_DIR / scp_name
    scp_path.write_text("".join(lines))
    print(f"  {orig_split:6s} → {scp_name:12s} ({len(lines)} files)")


📝 Tạo file SCP...
  train  → train.scp    (13500 files)
  valid  → dev.scp      (4500 files)
  test   → test.scp     (2250 files)


In [ ]:
# ============================================================================
# 5. CÀI ĐẶT THƯ VIỆN
# ============================================================================
print(f"\n📦 Cài đặt thư viện...")
!pip install -q yamlargparse librosa soundfile einops torchinfo rotary-embedding-torch tensorboard pesq pystoi


📦 Cài đặt thư viện...
  Preparing metadata (setup.py) ... done


In [ ]:
# ============================================================================
# 6. CONFIG
# ============================================================================
import yaml

config = {
    # === Chế độ ===
    'mode': 'train',
    'use_cuda': 1,

    # === Audio ===
    'sampling_rate': 8000,
    'max_length': 8,

    # === Model Architecture ===
    'network': 'MossFormer2_SS_8K',
    'num_spks': 2,

    # Encoder - Giữ nguyên
    'encoder_kernel_size': 16,
    'encoder_embedding_dim': 256,
    'mossformer_sequence_dim': 256,

    # Số layer
    'num_mossformer_layer': 16,

    # Recurrent
    'recurrent_type': 'fsmn',
    'recurrent_inner_channels': 128,

    # === Data Loading ===
    'load_type': 'one_input_multi_outputs',
    'tr_list': str(DATA_DIR / 'train.scp'),
    'cv_list': str(DATA_DIR / 'dev.scp'),
    'tt_list': str(DATA_DIR / 'test.scp'),

    # === Training - Điều chỉnh ===
    'init_learning_rate': 0.0001,       # 🔥 GIẢM LR (loss cao → cần LR thấp hơn)
    'finetune_learning_rate': 0.00003,
    'loss_threshold': -9999.0,
    'max_epoch': 30,                    # 🔥 GIẢM epochs (để test trước)
    'weight_decay': 0.0001,             # 🔥 TĂNG weight decay (chống overfit)
    'clip_grad_norm': 5.,               # 🔥 GIẢM clip (ổn định hơn)
    'seed': 777,

    # === DataLoader ===
    'num_workers': 4,                   # 🔥 GIẢM workers (tránh bottleneck CPU)
    'batch_size': 4,
    'accu_grad': 2,                     # 🔥 THÊM gradient accumulation
    'effec_batch_size': 16,             # 🔥 TĂNG effective batch size
}

# Lưu config
config_path = WORK_DIR / "config/train/vn_speechmix_lite_8s_8khz.yaml"
os.makedirs(config_path.parent, exist_ok=True)

with open(config_path, 'w') as f:
    yaml.safe_dump(config, f, sort_keys=False, default_flow_style=False)

print(f"\n⚙️ Config saved: {config_path}")
print(f"\n📄 Nội dung config:")
print(open(config_path).read())


⚙️ Config saved: /content/mossformer-mamba/mossformer2/config/train/vn_speechmix_lite_8s_8khz.yaml

📄 Nội dung config:
mode: train
use_cuda: 1
sampling_rate: 8000
max_length: 8
network: MossFormer2_SS_8K
num_spks: 2
encoder_kernel_size: 16
encoder_embedding_dim: 256
mossformer_sequence_dim: 256
num_mossformer_layer: 16
recurrent_type: fsmn
recurrent_inner_channels: 128
load_type: one_input_multi_outputs
tr_list: /content/mossformer-mamba/mossformer2/data/train.scp
cv_list: /content/mossformer-mamba/mossformer2/data/dev.scp
tt_list: /content/mossformer-mamba/mossformer2/data/test.scp
init_learning_rate: 0.0001
finetune_learning_rate: 3.0e-05
loss_threshold: -9999.0
max_epoch: 30
weight_decay: 0.0001
clip_grad_norm: 5.0
seed: 777
num_workers: 4
batch_size: 4
accu_grad: 2
effec_batch_size: 16



In [ ]:
# ============================================================================
# 7. TRAINING
# ============================================================================
%cd {WORK_DIR}

n_train = len(open(DATA_DIR / 'train.scp').readlines())
n_valid = len(open(DATA_DIR / 'dev.scp').readlines())
steps_per_epoch = n_train // config['batch_size']
save_freq = steps_per_epoch * 2

print(f"\n{'='*60}")
print(f"🚀 TRAINING MOSSFOMER2 LITE")
print(f"{'='*60}")
print(f"  Dataset: VN-SpeechMix 8s 8kHz")
print(f"  Model: MossFormer2 Lite")
print(f"  Train: {n_train} files | Valid: {n_valid} files")
print(f"  Batch size: {config['batch_size']}")
print(f"  Steps/epoch: ~{steps_per_epoch}")
print(f"  Max epochs: {config['max_epoch']}")
print(f"  Save: mỗi {save_freq} steps (~2 epochs)")
print(f"  Checkpoint: {CKPT_DIR}")
print(f"{'='*60}\n")

os.environ.update({
    'GPU_ID': '0',
    'N_GPU': '1',
    'CONFIG_PTH': str(config_path),
    'CHECKPOINT_DIR': str(CKPT_DIR),
    'TRAIN_FROM_LAST_CHECKPOINT': '1',
    'INIT_CHECKPOINT_PATH': 'None',
    'PRINT_FREQ': '50',
    'CHECKPOINT_SAVE_FREQ': str(save_freq),
})

!bash train.sh

print(f"\n{'='*60}")
print(f"🎉 TRAINING HOÀN THÀNH!")
print(f"  Checkpoints: {CKPT_DIR}")
print(f"  TensorBoard: tensorboard --logdir {CKPT_DIR}")
print(f"{'='*60}")

/content/mossformer-mamba/mossformer2

🚀 TRAINING MOSSFOMER2 LITE
  Dataset: VN-SpeechMix 8s 8kHz
  Model: MossFormer2 Lite
  Train: 13500 files | Valid: 4500 files
  Batch size: 4
  Steps/epoch: ~3375
  Max epochs: 30
  Save: mỗi 6750 steps (~2 epochs)
  Checkpoint: /content/drive/MyDrive/checkpoint-mossformer-lite-8s-8khz

2026-07-29 17:41:11.970111: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-29 17:41:12.037976: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
started on /content/drive/MyDrive/checkpoint-mo